In [1]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")


Using device: mps


In [2]:
model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Model num_labels: {model.config.num_labels}")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model: cross-encoder/ms-marco-MiniLM-L-6-v2
Model num_labels: 1


In [3]:
dataset = load_dataset("glue", "mrpc", split="validation")
print("Loaded dataset: glue/mrpc validation")
print(f"Original number of examples: {len(dataset)}")

labels_all = np.array(dataset["label"])
pos_indices = np.where(labels_all == 1)[0]
neg_indices = np.where(labels_all == 0)[0]

per_class_count = min(len(pos_indices), len(neg_indices), 75)
selected_indices = np.concatenate([neg_indices[:per_class_count], pos_indices[:per_class_count]])
selected_indices = np.sort(selected_indices)
balanced_dataset = dataset.select(selected_indices.tolist())

balanced_labels = np.array(balanced_dataset["label"])
num_neg = int((balanced_labels == 0).sum())
num_pos = int((balanced_labels == 1).sum())

print(f"Balanced slice size: {len(balanced_dataset)}")
print(f"Negative examples: {num_neg}")
print(f"Positive examples: {num_pos}")
print("Example row:")
print(balanced_dataset[0])


Loaded dataset: glue/mrpc validation
Original number of examples: 408
Balanced slice size: 150
Negative examples: 75
Positive examples: 75
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
batch_size = 32
labels = np.array(balanced_dataset["label"])
predictions = []
positive_probs = []
negative_probs = []
confidences = []
raw_scores = []

for start_idx in range(0, len(balanced_dataset), batch_size):
    batch = balanced_dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

        if logits.shape[-1] == 1:
            pos_prob = torch.sigmoid(logits.squeeze(-1))
            neg_prob = 1.0 - pos_prob
            probs = torch.stack([neg_prob, pos_prob], dim=-1)
        else:
            probs = torch.softmax(logits, dim=-1)
            if probs.shape[-1] < 2:
                raise ValueError(f"Unexpected logits shape: {logits.shape}")
            neg_prob = probs[:, 0]
            pos_prob = probs[:, 1]

        preds = (pos_prob >= 0.5).long()
        conf = torch.maximum(pos_prob, neg_prob)

    predictions.extend(preds.cpu().tolist())
    positive_probs.extend(pos_prob.cpu().tolist())
    negative_probs.extend(neg_prob.cpu().tolist())
    confidences.extend(conf.cpu().tolist())
    raw_scores.extend(logits.detach().cpu().numpy().tolist())

predictions = np.array(predictions)
positive_probs = np.array(positive_probs)
negative_probs = np.array(negative_probs)
confidences = np.array(confidences)

print(f"Completed inference for {len(predictions)} examples.")


Completed inference for 150 examples.


In [5]:
cm = confusion_matrix(labels, predictions, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

precision_by_class, recall_by_class, f1_by_class, support_by_class = precision_recall_fscore_support(
    labels,
    predictions,
    labels=[0, 1],
    average=None,
    zero_division=0
)

overall_accuracy = (predictions == labels).mean()
false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

print("Class-conditional evaluation metrics:")
print(f"Overall accuracy: {overall_accuracy:.4f}")
print()
print("Class 0: not_paraphrase")
print(f"  precision: {precision_by_class[0]:.4f}")
print(f"  recall   : {recall_by_class[0]:.4f}")
print(f"  f1       : {f1_by_class[0]:.4f}")
print(f"  support  : {int(support_by_class[0])}")
print()
print("Class 1: paraphrase")
print(f"  precision: {precision_by_class[1]:.4f}")
print(f"  recall   : {recall_by_class[1]:.4f}")
print(f"  f1       : {f1_by_class[1]:.4f}")
print(f"  support  : {int(support_by_class[1])}")
print()
print(f"False positive rate: {false_positive_rate:.4f}")
print(f"False negative rate: {false_negative_rate:.4f}")
print("Confusion matrix [[tn, fp], [fn, tp]]:")
print(cm)


Class-conditional evaluation metrics:
Overall accuracy: 0.5267

Class 0: not_paraphrase
  precision: 1.0000
  recall   : 0.0533
  f1       : 0.1013
  support  : 75

Class 1: paraphrase
  precision: 0.5137
  recall   : 1.0000
  f1       : 0.6787
  support  : 75

False positive rate: 0.9467
False negative rate: 0.0000
Confusion matrix [[tn, fp], [fn, tp]]:
[[ 4 71]
 [ 0 75]]


In [6]:
label_map = {0: "not_paraphrase", 1: "paraphrase"}

false_positive_indices = np.where((labels == 0) & (predictions == 1))[0]
false_negative_indices = np.where((labels == 1) & (predictions == 0))[0]

fp_sorted = false_positive_indices[np.argsort(-positive_probs[false_positive_indices])] if len(false_positive_indices) > 0 else np.array([], dtype=int)
fn_sorted = false_negative_indices[np.argsort(-negative_probs[false_negative_indices])] if len(false_negative_indices) > 0 else np.array([], dtype=int)

top_k = 5

def print_example(idx, title):
    row = balanced_dataset[int(idx)]
    true_label = int(labels[idx])
    pred_label = int(predictions[idx])
    print(title)
    print(f"index      : {idx}")
    print(f"sentence1  : {row['sentence1']}")
    print(f"sentence2  : {row['sentence2']}")
    print(f"true label : {true_label} ({label_map[true_label]})")
    print(f"pred label : {pred_label} ({label_map[pred_label]})")
    print(f"p(class=0) : {negative_probs[idx]:.4f}")
    print(f"p(class=1) : {positive_probs[idx]:.4f}")
    print(f"confidence : {confidences[idx]:.4f}")
    print("-" * 80)

print("Most confident false positives:")
if len(fp_sorted) == 0:
    print("None")
else:
    for rank, idx in enumerate(fp_sorted[:top_k], start=1):
        print_example(idx, f"False positive #{rank}")

print("Most confident false negatives:")
if len(fn_sorted) == 0:
    print("None")
else:
    for rank, idx in enumerate(fn_sorted[:top_k], start=1):
        print_example(idx, f"False negative #{rank}")


Most confident false positives:
False positive #1
index      : 85
sentence1  : Shattered Glass , " starring Hayden Christensen as Stephen Glass , debuted well with $ 80,000 in eight theaters .
sentence2  : " Shattered Glass " _ starring Hayden Christensen as Stephen Glass , The New Republic journalist fired for fabricating stories _ debuted well with $ 80,000 in eight theaters .
true label : 0 (not_paraphrase)
pred label : 1 (paraphrase)
p(class=0) : 0.0000
p(class=1) : 1.0000
confidence : 1.0000
--------------------------------------------------------------------------------
False positive #2
index      : 124
sentence1  : Morrill 's wife , Ellie , sobbed and hugged Bondeson 's sister-in-law during the service .
sentence2  : At the service Morrill 's widow , Ellie , sobbed and hugged Bondeson 's sister-in-law as people consoled her .
true label : 0 (not_paraphrase)
pred label : 1 (paraphrase)
p(class=0) : 0.0000
p(class=1) : 1.0000
confidence : 1.0000
----------------------------------

In [7]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print("subset=balanced fixed slice")
print(f"device={device}")
print(f"num_examples={len(balanced_dataset)}")
print(f"num_negative={num_neg}")
print(f"num_positive={num_pos}")
print(f"accuracy={overall_accuracy:.4f}")
print(f"precision_class_0={precision_by_class[0]:.4f}")
print(f"recall_class_0={recall_by_class[0]:.4f}")
print(f"f1_class_0={f1_by_class[0]:.4f}")
print(f"precision_class_1={precision_by_class[1]:.4f}")
print(f"recall_class_1={recall_by_class[1]:.4f}")
print(f"f1_class_1={f1_by_class[1]:.4f}")
print(f"false_positive_rate={false_positive_rate:.4f}")
print(f"false_negative_rate={false_negative_rate:.4f}")
print(f"false_positives={len(false_positive_indices)}")
print(f"false_negatives={len(false_negative_indices)}")


RESULT SUMMARY
model=cross-encoder/ms-marco-MiniLM-L-6-v2
dataset_split=glue/mrpc validation
subset=balanced fixed slice
device=mps
num_examples=150
num_negative=75
num_positive=75
accuracy=0.5267
precision_class_0=1.0000
recall_class_0=0.0533
f1_class_0=0.1013
precision_class_1=0.5137
recall_class_1=1.0000
f1_class_1=0.6787
false_positive_rate=0.9467
false_negative_rate=0.0000
false_positives=71
false_negatives=0
